## Import Libraries

In [123]:
import pandas as pd 
import numpy as np
from pathlib import Path

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC, LinearSVC

In [124]:
# Run once

import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/vishalbhaga/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/vishalbhaga/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/vishalbhaga/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/vishalbhaga/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

## Load the Data

In [125]:
def load_data(folder_path):
    texts = []
    labels = []
    path_names = []

    for label in ['pos', 'neg']:
        for file_path in Path(folder_path, label).glob("*.txt"):
            texts.append(file_path.read_text(encoding='latin-1'))       # handles characters that break UTF-8
            labels.append(1 if label == 'pos' else 0)
            path_names.append(str(file_path))
    
    return [texts, labels, path_names]

In [126]:
input_data = load_data("review_polarity/txt_sentoken")

df = pd.DataFrame({
    "text": input_data[0],
    "label": input_data[1]
})

df.head()

,text,label
0,assume nothing . \nthe phrase is perhaps one o...,1
1,plot : derek zoolander is a male model . \nhe ...,1
2,i actually am a fan of the original 1961 or so...,1
3,a movie that's been as highly built up as the ...,1
4,""" good will hunting "" is two movies in one : ...",1


In [127]:
df['label'].value_counts()

label
1    1000
0    1000
Name: count, dtype: int64

## Data Preprocessing

In [128]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Custom function to handle common parts of speech
def get_part_of_speech(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

def pre_processing(text):
    # Lowercasing
    text = text.lower()

    # Tokenisation - split text into words
    words = word_tokenize(text)

    # Remove Stopwords - remove common words such as "I", "am" etc
    words = [w for w in words if w not in stop_words]

    # Lemmatisation - reduce words to base form
    # By default only works on nouns
    tagged = pos_tag(words)
    words = [
        lemmatizer.lemmatize(word, get_part_of_speech(tag))
        for word, tag in tagged
    ]

    return " ".join(words)

In [129]:
df["pre_processed"] = df["text"].apply(pre_processing)

In [130]:
df.head()

,text,label,pre_processed
0,assume nothing . \nthe phrase is perhaps one o...,1,assume nothing . phrase perhaps one use 1990 '...
1,plot : derek zoolander is a male model . \nhe ...,1,plot : derek zoolander male model . also dumb ...
2,i actually am a fan of the original 1961 or so...,1,actually fan original 1961 live-action-disney ...
3,a movie that's been as highly built up as the ...,1,"movie 's highly build truman show , review boa..."
4,""" good will hunting "" is two movies in one : ...",1,`` good hunting `` two movie one : independent...


In [131]:
# Split data into train, validation and test sets

X = df['pre_processed']
y = df['label']

# First split: 70% Train, 30% Temp (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 2. Second split: Split the 30% Temp into two 15% sets (Val and Test)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

In [132]:
# Check class counts in each split
print(f'Train set class count: \n{pd.Series(y_train).value_counts()}')
print(f'Validation set class count: \n{pd.Series(y_val).value_counts()}')
print(f'Test set class count: \n{pd.Series(y_test).value_counts()}')

Train set class count: 
label
0    700
1    700
Name: count, dtype: int64
Validation set class count: 
label
1    150
0    150
Name: count, dtype: int64
Test set class count: 
label
1    150
0    150
Name: count, dtype: int64


## Make Model

In [133]:
def make_model(feature_extraction = 'tfidf', svm_type = 'linear', penalty = 'l2', C=1):
    if feature_extraction == 'word2vec':
        featExtr = TfidfVectorizer()
    else:
        featExtr = TfidfVectorizer()

    if svm_type == 'rbf':
        model = SVC(kernel='rbf', C=C)
    else:
        model = LinearSVC(penalty=penalty)
    
    pipeline = Pipeline([
        ('featExtr', featExtr),
        ('model', model)
    ])

    return pipeline

## Feature Extraction Analysis

In [135]:
model = make_model(feature_extraction='tfidf')

model.fit(X_train, y_train)
tfidf = model.named_steps['featExtr']

vocabulary_size = len(tfidf.vocabulary_)
print("Vocabulary size:", vocabulary_size)

print("Max DF:", tfidf.max_df)

X_train_tfidf = tfidf.transform(X_train)
print(X_train_tfidf.shape)

Vocabulary size: 29250
Max DF: 1.0
(1400, 29250)
